# Longitudinal Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Anna Ringwood
- AI Acknowledgements: No AI has been used thus far, but this file is still a work-in-progress.

## Text Preprocessing — Processing Text from JOB Minutes

This notebook contains code for ingesting the scraped text files and beginning to further extract only relevant text information.

In [ ]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse

In [2]:
# Set up paths, similar to how we did in the preceding file
BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where the raw text files are kept

In [3]:
raw_docs = []
doc_names = []

# This section's code taken from last block of preceding file with minor modifications to variable names
for path in sorted(OUT_DIR.glob("*.txt")):
    raw_docs.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(raw_docs)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


In [4]:
import spacy
nlp = spacy.load('en_core_web_sm', disable = ['parser', 'ner']) # disable parser and NER so package runs faster

In [ ]:
from tqdm import tqdm

documents = []
for raw_doc in tqdm(raw_docs): # use tqdm for progress monitoring; time varies depending on your machine: ~6-15 min
    documents.append(nlp(raw_doc))

  0%|          | 0/156 [00:00<?, ?it/s]

100%|██████████| 156/156 [06:29<00:00,  2.50s/it]


### Removing Unneeded Text Data

Now that the text files have been imported and somewhat cleaned, let's clean them up further. For example, the more recent documents come with additional vocab glossaries and indices, which would both throw off our analysis and take up unnecessary processing space during lemmatization. We'll try removing everything after the word "ADJOURNMENT" in each document. First, let's make sure that each document only has one instance of "ADJOURNMENT."

In [7]:
current_doc_index = 0
one_adjourn_idxs = []
gt_one_adjourn_idxs = []
lt_one_adjourn_idxs = []

for doc in raw_docs:

    if doc.upper().count("ADJOURNMENT") > 1:
        gt_one_adjourn_idxs.append(current_doc_index)
    elif doc.upper().count("ADJOURNMENT") < 1:
        lt_one_adjourn_idxs.append(current_doc_index)
    else:
        one_adjourn_idxs.append(current_doc_index)

    current_doc_index += 1

In [9]:
print(f"One instance of adjourn ({len(one_adjourn_idxs)}): {one_adjourn_idxs}")
print(f"Less than one instance of adjourn ({len(lt_one_adjourn_idxs)}): {lt_one_adjourn_idxs}")
print(f"More than one instance of adjourn ({len(gt_one_adjourn_idxs)}): {gt_one_adjourn_idxs}")

One instance of adjourn (92): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 28, 29, 30, 31, 34, 35, 36, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 92, 93, 96, 97, 98, 99, 101, 102, 106, 107, 108, 109, 110, 111, 112, 113, 114]
Less than one instance of adjourn (39): [25, 26, 27, 32, 33, 37, 38, 39, 40, 41, 42, 43, 44, 91, 94, 95, 100, 104, 105, 115, 116, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 132, 134, 135, 136, 139, 140, 144, 147]
More than one instance of adjourn (25): [18, 67, 86, 103, 117, 118, 123, 130, 131, 133, 137, 138, 141, 142, 143, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155]


In [13]:
[doc_names[i] for i in gt_one_adjourn_idxs]

['2014_1_0_JOB_Minutes.txt',
 '2018_8_0_JOB_Minutes.txt',
 '2020_5_0_JOB_Minutes.txt',
 '2022_10_0_JOB_Minutes.txt',
 '2023_12_0_JOB_Minutes.txt',
 '2023_1_0_JOB_Minutes.txt',
 '2023_6_0_JOB_Minutes.txt',
 '2024_1_0_JOB_Minutes.txt',
 '2024_2_0_JOB_Minutes.txt',
 '2024_3_0_JOB_Minutes.txt',
 '2024_6_0_JOB_Minutes.txt',
 '2024_7_0_JOB_Minutes.txt',
 '2025_10_0_JOB_Minutes.txt',
 '2025_11_0_JOB_Minutes.txt',
 '2025_12_0_JOB_Minutes.txt',
 '2025_2_0_JOB_Minutes.txt',
 '2025_3_0_JOB_Minutes.txt',
 '2025_5_0_JOB_Minutes.txt',
 '2025_6_0_JOB_Minutes.txt',
 '2025_7_0_JOB_Minutes.txt',
 '2025_8_0_JOB_Minutes.txt',
 '2025_8_1_JOB_Minutes.txt',
 '2025_9_0_JOB_Minutes.txt',
 '2026_1_0_JOB_Minutes.txt',
 '2026_2_0_JOB_Minutes.txt']

Most of the documents do only have one occurrence of the word "adjournment."

Of the documents that have more than one occurrence of "adjournment," most of them appear to be from more recent years, but not as consistently as we would expect. Some manual examination of the documents is probably needed.